# OE 2026 - dual energy CRLB

In [ ]:
import xpci_crlb as crlb
from xpci_simulate import Material, apply_psf, get_wavelen

import os
from time import time

import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from flax import linen as nn
import dataclasses
import chromatix.functional as cx
from chromatix.ops import init_plane_resample

# Use float64 needed for smooth MLE loss; explicitly set float32 elsewhere
jax.config.update('jax_enable_x64', True)
JTYPE = jnp.float32
DTYPE = np.float32
CTYPE = jnp.complex64

In [ ]:
# Basis materials
tissue = Material('tissue', 'H(10.2)C(14.3)N(3.4)O(70.8)Na(0.2)P(0.3)S(0.3)Cl(0.2)K(0.3)', 1.06)
bone = Material('bone', 'H(3.4)C(15.5)N(4.2)O(43.5)Na(0.1)Mg(0.2)P(10.3)S(0.3)Ca(22.5)', 1.92)
adip = Material('adipose', 'H(11.2)C(61.9)N(1.7)O(25.1)', 0.9) 
gland = Material('gland', 'H(10.2)C(18.4)N(3.2)O(67.6)', 1.1)  

# 1. Dual-energy CRLB$(E_1, E_2)$ heatmaps

In [ ]:
# Parameters

MBs = [[bone, tissue], 
       [gland, adip]]

I0_per_m2 = 1e3 * 1e12  # 1e3 incident photons per micron
fov = 64e-6
NX = 64
T = 5e-3
F = 0.2
pars = dict(
    R = 100e-3,
    T1 = F*T,       
    T2 = (1-F)*T,  
    obj_radius = 15e-6,
    psf = 'lorentzian',
    Nx = NX,
    dx = fov / NX,
    fwhm = 2*fov / NX,
    pshift = 0.2
)

## 1.1. mono-CRLB

In [ ]:
%%time

outd_main = 'outputs/de_data/'
os.makedirs(outd_main, exist_ok=True)

###################################
### Parameters

pars['psf'] = 'lorentzian'  # just in case

E_min = 8
E_max = 22
dE = 1
energies = np.arange(E_min, E_max+dE, dE)

pairs = [(ii, jj) for ii in range(energies.size) for jj in range(ii, energies.size)]

# Save
np.array(energies).astype(DTYPE).tofile(outd_main+'energies.bin')


###################################
### Run

for MB in MBs:

    outd = outd_main + f'{MB[0].name}_{MB[1].name}/'
    os.makedirs(outd + 'hats', exist_ok=True)

    pars['mat1'] = MB[0]
    pars['mat2'] = MB[1]
    
    
    # ---------------- CRLB ----------------
    crlb_grid1, crlb_grid2 = np.zeros((2, energies.size, energies.size), dtype=DTYPE)

    for ii, jj in pairs:
        E1 = float(energies[ii])
        E2 = float(energies[jj])
        cvars, csnrs = crlb.compute_crlb_mono(E1=E1, E2=E2, I0_per_m2=I0_per_m2, **pars)
        crlb_grid1[ii, jj] = crlb_grid1[jj, ii] = DTYPE(cvars[0])
        crlb_grid2[ii, jj] = crlb_grid2[jj, ii] = DTYPE(cvars[1])

    np.array(crlb_grid1).astype(DTYPE).tofile(outd+'crlb_grid1.bin')
    np.array(crlb_grid2).astype(DTYPE).tofile(outd+'crlb_grid2.bin')

    
    # ---------------- plot ----------------
    fig, ax = plt.subplots(1, 2, figsize=[8,3], layout='constrained', dpi=300)
    fig.suptitle(f'{MB[0].name} / {MB[1].name} basis -- CRLB')
    
    for j, crlb_grid in enumerate([crlb_grid1,crlb_grid2]):
        m = ax[j].imshow(crlb_grid)
        fig.colorbar(m, ax=ax[j])
        ax[j].set_xticks([])
        ax[j].set_yticks([])
        
    plt.show()
    

## 1.2. mono-MLE

In [ ]:
# Optimized DE simulation class

class DualEnergyPBI(nn.Module):

    material_basis: tuple = dataclasses.field(default_factory=lambda: (bone, tissue))  # update!

    propdist = pars['R']

    # Phantom
    obj_radius = pars['obj_radius']
    pshift = pars['pshift']

    # Detector
    det_fwhm = pars['fwhm']
    det_psf = pars['psf']
    resampling_method: str = 'linear'
    I0 = I0_per_m2
    det_dx = pars['dx']
    det_Nx = pars['Nx']
    det_fov = det_dx * det_Nx

    # Upsampled-sim
    phantom_dx = 0.5e-6
    phantom_Nx = int(det_fov / phantom_dx) + 1
    phantom_fov = phantom_dx * phantom_Nx

    # Misc.
    N_pad: int = 8
    n_medium: float = 1
    cval = jnp.array(1 + 0j, dtype=CTYPE)

    def setup(self):

        self._det_dx_f = float(self.det_dx)
        self._det_fwhm_f = float(self.det_fwhm)

        self._phantom_dx = jnp.asarray(self.phantom_dx, dtype=JTYPE)
        self._det_dx = jnp.asarray(self.det_dx, dtype=JTYPE)
        self._scale_resample = (self._phantom_dx / self._det_dx)**2
        self._det_dx2 = self._det_dx**2
        self._I0 = jnp.asarray(self.I0, dtype=JTYPE)
        self._propdist = jnp.asarray(self.propdist, dtype=JTYPE)
        self._n_medium = jnp.asarray(self.n_medium, dtype=JTYPE)

        # Construct cylinder basis masks
        x = jnp.linspace(-self.phantom_fov / 2, self.phantom_fov / 2, self.phantom_Nx, dtype=JTYPE)
        X, Y = jnp.meshgrid(x, x)
        shift_dist = jnp.asarray(self.pshift * self.obj_radius, dtype=JTYPE)

        self.tmasks = jnp.array([
            ((X - shift_dist)**2 + (Y - shift_dist)**2) < (self.obj_radius**2),
            ((X + shift_dist)**2 + (Y + shift_dist)**2) < (self.obj_radius**2)
        ], dtype=JTYPE)

        # Resample propagated wave -> detector pixel size
        self.det_resample_func = init_plane_resample(
            (self.det_Nx, self.det_Nx),
            (self.det_dx, self.det_dx),
            resampling_method=self.resampling_method
        )

    def __call__(self, E1: float, E2: float, T1: float, T2: float):

        E1 = jnp.asarray(E1, dtype=JTYPE)
        E2 = jnp.asarray(E2, dtype=JTYPE)
        T1 = jnp.asarray(T1, dtype=JTYPE)
        T2 = jnp.asarray(T2, dtype=JTYPE)

        energies = jnp.stack([E1, E2])  # (2,)
        wavelens = jnp.asarray(get_wavelen(energies), dtype=JTYPE)

        ds1, bs1 = self.material_basis[0].delta_beta(energies)
        ds2, bs2 = self.material_basis[1].delta_beta(energies)
        ds1 = jnp.asarray(ds1, dtype=JTYPE); bs1 = jnp.asarray(bs1, dtype=JTYPE)
        ds2 = jnp.asarray(ds2, dtype=JTYPE); bs2 = jnp.asarray(bs2, dtype=JTYPE)

        tmap1 = T1 * self.tmasks[0]
        tmap2 = T2 * self.tmasks[1]

        dn_proj = (tmap1[..., None] * ds1[None, None, :]) + (tmap2[..., None] * ds2[None, None, :])
        beta_proj = (tmap1[..., None] * bs1[None, None, :]) + (tmap2[..., None] * bs2[None, None, :])

        field = cx.plane_wave(
            shape=(self.phantom_Nx, self.phantom_Nx),
            dx=self._phantom_dx,
            spectrum=wavelens,
            spectral_density=jnp.ones(2, dtype=JTYPE)
        )
        field = field / field.amplitude.max()

        exit_field = cx.thin_sample(
            field,
            beta_proj[None, ..., None],
            dn_proj[None, ..., None],
            jnp.asarray(1.0, dtype=JTYPE)
        )

        det_field = cx.transfer_propagate(
            exit_field,
            self._propdist,
            self._n_medium,
            self.N_pad,
            cval=self.cval,
            mode='same'
        )

        det_intensity = det_field.amplitude.squeeze()**2  # (Nx, Nx, 2)

        imgs = self.det_resample_func(
            det_intensity[..., None],
            field.dx.ravel()[:1]
        )[..., 0] * self._scale_resample  # (det_Nx, det_Nx, 2)

        # PSF per energy channel (avoid tracer dx/fwhm)
        imgs = jax.vmap(
            lambda img: apply_psf(
                img, self._det_dx_f,
                psf=self.det_psf, fwhm=self._det_fwhm_f,
                kernel_width=8.0
            ).astype(JTYPE),
            in_axes=-1,
            out_axes=-1
        )(imgs)

        imgs = (self._I0 * self._det_dx2) * imgs
        return imgs


In [ ]:
# MLE helpers

def nll64_to32(y_k, data):
    y64 = jnp.asarray(y_k, dtype=jnp.float64)
    d64 = jnp.asarray(data, dtype=jnp.float64)
    y64 = jnp.clip(y64, 1e-12, None)
    out64 = jnp.mean(y64 - d64 * jnp.log(y64), dtype=jnp.float64)
    return out64.astype(JTYPE)

def make_forward(model, vars_):
    def f(E1, E2, T1, T2):
        return model.apply(vars_, E1, E2, T1, T2)
    return jax.jit(jax.vmap(f, in_axes=(None, None, 0, 0)))

In [ ]:
%%time

N_reps = 1000
N_batches = 10
key = jax.random.PRNGKey(3)  
key_init, key_noise = jax.random.split(key)

T1_gt = DTYPE(pars['T1'])
T2_gt = DTYPE(pars['T2'])

N_grid = 20
dT = DTYPE(1e-7)
T1_lim = (T1_gt - dT * (N_grid / 2), T1_gt + dT * (N_grid / 2))
T2_lim = (T2_gt - dT * (N_grid / 2), T2_gt + dT * (N_grid / 2))

T1s = jnp.linspace(T1_lim[0], T1_lim[1], N_grid, dtype=JTYPE)
T2s = jnp.linspace(T2_lim[0], T2_lim[1], N_grid, dtype=JTYPE)
T1_grid, T2_grid = jnp.meshgrid(T1s, T2s)
np.array(T1s).astype(DTYPE).tofile(outd_main+'T1s.bin')
np.array(T2s).astype(DTYPE).tofile(outd_main+'T2s.bin')

T1_gt_j = jnp.asarray([T1_gt], dtype=JTYPE)
T2_gt_j = jnp.asarray([T2_gt], dtype=JTYPE)

@jax.jit
def mle_batch(key, y_grid, data_ideal):
    keys = jax.random.split(key, N_reps + 1)
    key_next = keys[0]
    keys_rep = keys[1:]

    data_reps = jax.vmap(lambda k: jax.random.poisson(k, data_ideal).astype(JTYPE))(keys_rep)

    def nll_grid_one(data_rep):
        nll_ = lambda y_k: nll64_to32(y_k, data_rep)
        return jax.vmap(jax.vmap(nll_, in_axes=0), in_axes=0)(y_grid)

    nll_grids = jax.vmap(nll_grid_one)(data_reps)  # (N_reps, N_grid, N_grid)

    flat = nll_grids.reshape(N_reps, -1)
    idx = jnp.argmin(flat, axis=1)

    i1 = idx // nll_grids.shape[2]
    i2 = idx %  nll_grids.shape[2]
    return key_next, i1, i2

    
###################################
### Run

for MB in MBs:

    outd = outd_main + f'{MB[0].name}_{MB[1].name}/'
    os.makedirs(outd + 'hats', exist_ok=True)

    pars['mat1'] = MB[0]
    pars['mat2'] = MB[1]
    
    print(f'{MB[0].name} / {MB[1].name} basis')

    # ---------------- MLE ----------------
    mle_grid1, mle_grid2 = np.zeros((2, energies.size, energies.size), dtype=DTYPE)
    
    model = DualEnergyPBI(material_basis=MB)
    vars_ = model.init(key_init, JTYPE(10.0), JTYPE(20.0), JTYPE(0.0), JTYPE(0.0))
    forward = make_forward(model, vars_)

    t0 = time()
    # for ii, jj in pairs:
        # print(f'({energies[ii]:.1f}, {energies[jj]:.1f} keV) --- {time() - t0:.1f} s')

    for ind, pair in enumerate(pairs):
        ii, jj = pair
        if (ind%10 == 0):
            print(f'i = {ind}/{len(pairs)}: ({energies[ii]:.1f}, {energies[jj]:.1f} keV) --- {time() - t0:.1f} s')

        E1 = jnp.asarray(energies[ii], dtype=JTYPE)
        E2 = jnp.asarray(energies[jj], dtype=JTYPE)

        # Ideal data
        data_ideal = forward(E1, E2, T1_gt_j, T2_gt_j).squeeze().astype(JTYPE)

        # Grid predictions (N_grid^2 forward evals)
        y_grid_flat = forward(E1, E2, T1_grid.ravel(), T2_grid.ravel()).astype(JTYPE)
        y_grid = y_grid_flat.reshape(T2s.size, T1s.size, *y_grid_flat.shape[1:])
        y_grid = jnp.swapaxes(y_grid, 0, 1)  # (N_grid, N_grid, ...)

        T1_hats_tot = np.zeros(N_batches * N_reps, dtype=DTYPE)
        T2_hats_tot = np.zeros(N_batches * N_reps, dtype=DTYPE)
        
        for kk in range(N_batches):            
            key_noise, i1s, i2s = mle_batch(key_noise, y_grid, data_ideal)
    
            T1_hats = np.array(T1s[i1s], dtype=DTYPE)
            T2_hats = np.array(T2s[i2s], dtype=DTYPE)

            T1_hats_tot[kk*N_reps:(kk+1)*N_reps] = T1_hats
            T2_hats_tot[kk*N_reps:(kk+1)*N_reps] = T2_hats
            
            np.array(T1_hats).astype(DTYPE).tofile(outd+f'hats/T1s_{E1:.1f}_{E2:.1f}_{N_reps}_{kk}.bin')
            np.array(T2_hats).astype(DTYPE).tofile(outd+f'hats/T2s_{E1:.1f}_{E2:.1f}_{N_reps}_{kk}.bin')

        mle_grid1[ii, jj] = mle_grid1[jj, ii] = np.var(T1_hats_tot).astype(DTYPE)
        mle_grid2[ii, jj] = mle_grid2[jj, ii] = np.var(T2_hats_tot).astype(DTYPE)

    np.array(mle_grid1).astype(DTYPE).tofile(outd+'mle_grid1.bin')
    np.array(mle_grid2).astype(DTYPE).tofile(outd+'mle_grid2.bin')


    # ---------------- plot ----------------
    fig, ax = plt.subplots(1, 2, figsize=[8,3], layout='constrained', dpi=300)
    fig.suptitle(f'{MB[0].name} / {MB[1].name} basis -- MLE')
    
    for j, mle_grid in enumerate([mle_grid1,mle_grid2]):
        m = ax[j].imshow(mle_grid)
        fig.colorbar(m, ax=ax[j])
        ax[j].set_xticks([])
        ax[j].set_yticks([])
        
    plt.show()


# 2. Polychromatic CRLB vs. $E_t$

In [ ]:
# Load spectrum 
fname = 'inputs/tasmics_30kVp_05mmAl.csv'

with open(fname, 'r') as f:
    energies, counts = np.array([line.split(',') for line in f.readlines()[1:]], dtype=DTYPE).T
    counts /= np.max(counts)  

# Include only significant nonzero components
cmin = 0.01
energies = energies[counts > cmin]
counts = counts[counts > cmin]

# Use sufficiently small dE (for T = 5 mm, dE <= 0.05 keV is good)
dE_spec = 0.05
energies2 = np.arange(energies.min(), energies.max()+dE_spec, dE_spec).astype(DTYPE)
counts2 = np.interp(energies2, energies, counts).astype(DTYPE)
spectrum = np.array([energies2, counts2 / counts2.sum()])

# Helper to split and truncate spectra about some threshold energy
def get_specs_truncated(energies, counts, Et, gap_keV=0, fwhm_keV=5):
    c1, c2 = crlb.split_spectrum_gaussian(energies, counts, Et, gap_keV, fwhm_keV)
    spec1 = jnp.array([energies, c1], dtype=JTYPE)
    spec2 = jnp.array([energies, c2], dtype=JTYPE)
    return spec1, spec2

# Show
fig, ax = plt.subplots(figsize=[7,3], layout='constrained', dpi=300)
ax.set_ylabel('normalized counts')
ax.set_xlabel('energy [keV]')
ax.plot(spectrum[0], spectrum[1], 'b.-', ms=3, alpha=0.3)
plt.show()

## 2.1. poly-CRLB

In [ ]:
%%time

outd_main = 'outputs/poly_data/'
os.makedirs(outd_main, exist_ok=True)


###################################
### Parameters

pars['psf'] = 'gaussian'  # PCD has Gaussian PSF, otherwise inherit pars from DE

dE = 0.2
threshs = np.arange(5, 35+dE, dE)    
np.array(threshs).astype(DTYPE).tofile(outd_main+'threshs.bin')

energies = spectrum[0]
counts = 2 * I0_per_m2 * spectrum[1]


###################################
### Run

for MB in MBs:

    outd = outd_main + f'{MB[0].name}_{MB[1].name}/'
    os.makedirs(outd, exist_ok=True)

    pars['mat1'] = MB[0]
    pars['mat2'] = MB[1]
    
    # ---------------- CRLB ----------------
    crlb_vars1, crlb_vars2 = np.zeros([2, len(threshs)])
    
    for ii, Et in enumerate(threshs):        
        spec1, spec2 = get_specs_truncated(energies, counts, Et)
        cvars, csnrs = crlb.compute_crlb_spectral(
            spectrum1=spec1, spectrum2=spec2, **pars
        )
        crlb_vars1[ii] = cvars[0]
        crlb_vars2[ii] = cvars[1]

    np.array(crlb_vars1).astype(DTYPE).tofile(outd+'crlb_vars1.bin')
    np.array(crlb_vars2).astype(DTYPE).tofile(outd+'crlb_vars2.bin')
    
    # ---------------- plot ----------------
    fig, ax = plt.subplots(1, 2, figsize=[7,2.8], layout='constrained', dpi=300)
    
    fig.suptitle(f'{MB[0].name} / {MB[1].name} basis')
    for j, crlb_vars in enumerate([crlb_vars1,crlb_vars2]):
        ax[j].plot(threshs, crlb_vars, '.-')
    plt.show()
    

## 2.2. poly-MLE

In [ ]:
# Optimized polychromatic simulation class

class SpectralPBI(nn.Module):

    spectrum1: jnp.ndarray
    spectrum2: jnp.ndarray
    material_basis: tuple = dataclasses.field(default_factory=lambda: (bone, tissue))

    propdist = pars['R']

    # Phantom
    obj_radius = pars['obj_radius']
    pshift = pars['pshift']

    # Detector
    det_fwhm = pars['fwhm']
    det_psf = pars['psf']
    resampling_method: str = 'linear'
    det_dx = pars['dx']
    det_Nx = pars['Nx']
    det_fov = det_dx * det_Nx

    phantom_dx = 1e-6
    phantom_Nx = int(det_fov / phantom_dx) + 1
    phantom_fov = phantom_dx * phantom_Nx

    # Misc.
    N_pad: int = 8
    n_medium: float = 1
    cval = jnp.array(1 + 0j, dtype=CTYPE)

    def setup(self):

        self._phantom_dx = jnp.asarray(self.phantom_dx, dtype=JTYPE)
        self._det_dx = jnp.asarray(self.det_dx, dtype=JTYPE)
        self._propdist = jnp.asarray(self.propdist, dtype=JTYPE)
        self._n_medium = jnp.asarray(self.n_medium, dtype=JTYPE)

        self._scale_resample = (self._phantom_dx / self._det_dx)**2
        self._det_dx2 = self._det_dx**2

        # concrete for apply_psf -> jnp.arange
        self._det_dx_f = float(self.det_dx)
        self._det_fwhm_f = float(self.det_fwhm)

        x = jnp.linspace(-self.phantom_fov / 2, self.phantom_fov / 2, self.phantom_Nx, dtype=JTYPE)
        X, Y = jnp.meshgrid(x, x)
        shift_dist = jnp.asarray(self.pshift * self.obj_radius, dtype=JTYPE)

        self.tmasks = jnp.array([
            ((X - shift_dist)**2 + (Y - shift_dist)**2) < (self.obj_radius**2),
            ((X + shift_dist)**2 + (Y + shift_dist)**2) < (self.obj_radius**2)
        ], dtype=JTYPE)

        self.det_resample_func = init_plane_resample(
            (self.det_Nx, self.det_Nx),
            (self.det_dx, self.det_dx),
            resampling_method=self.resampling_method
        )

    def __call__(self, T1: float, T2: float):

        T1, T2 = jnp.asarray(T1, dtype=JTYPE), jnp.asarray(T2, dtype=JTYPE)

        tmap1 = T1 * self.tmasks[0]
        tmap2 = T2 * self.tmasks[1]

        imgs = jnp.zeros((2, self.det_Nx, self.det_Nx), dtype=JTYPE)

        for i, spectrum in enumerate((self.spectrum1, self.spectrum2)):

            energies = jnp.asarray(spectrum[0], dtype=JTYPE)
            counts = jnp.asarray(spectrum[1], dtype=JTYPE)

            ds1, bs1 = self.material_basis[0].delta_beta(energies)
            ds2, bs2 = self.material_basis[1].delta_beta(energies)
            ds1, bs1 = jnp.asarray(ds1, dtype=JTYPE), jnp.asarray(bs1, dtype=JTYPE)
            ds2, bs2 = jnp.asarray(ds2, dtype=JTYPE), jnp.asarray(bs2, dtype=JTYPE)

            dn_proj = tmap1[..., None] * ds1[None, None, :] + tmap2[..., None] * ds2[None, None, :]
            beta_proj = tmap1[..., None] * bs1[None, None, :] + tmap2[..., None] * bs2[None, None, :]

            field = cx.plane_wave(
                shape=(self.phantom_Nx, self.phantom_Nx),
                dx=self._phantom_dx,
                spectrum=jnp.asarray(get_wavelen(energies), dtype=JTYPE),
                spectral_density=jnp.ones(energies.size, dtype=JTYPE)
            )
            field = field / field.amplitude.max()

            exit_field = cx.thin_sample(
                field,
                beta_proj[None, ..., None],
                dn_proj[None, ..., None],
                JTYPE(1.0)
            )

            det_field = cx.transfer_propagate(
                exit_field,
                self._propdist,
                self._n_medium,
                self.N_pad,
                cval=self.cval,
                mode='same'
            )

            det_intensity = det_field.amplitude.squeeze()**2

            imgs_E = self.det_resample_func(
                det_intensity[..., None],
                field.dx.ravel()[:1]
            )[..., 0] * self._scale_resample

            img_polychrom = jnp.sum(imgs_E * counts[None, None, :], axis=-1)

            img_polychrom = apply_psf(
                img_polychrom, self._det_dx_f, psf=self.det_psf, fwhm=self._det_fwhm_f, kernel_width=8.0
            ).astype(JTYPE)

            imgs = imgs.at[i].set(img_polychrom * self._det_dx2)

        return imgs

In [ ]:
%%time

N_reps = 1000
N_batches = 10
key = jax.random.PRNGKey(3)
key_init, key_noise = jax.random.split(key)

T1_gt = jnp.asarray([pars['T1']], dtype=JTYPE)
T2_gt = jnp.asarray([pars['T2']], dtype=JTYPE)

@jax.jit
def mle_batch(key, y_grid, data_ideal):
    keys = jax.random.split(key, N_reps + 1)
    key_next = keys[0]
    keys_rep = keys[1:]

    data_reps = jax.vmap(lambda k: jax.random.poisson(k, data_ideal).astype(JTYPE))(keys_rep)

    def nll_grid_one(data_rep):
        nll_ = lambda y_k: nll64_to32(y_k, data_rep)
        return jax.vmap(jax.vmap(nll_, in_axes=0), in_axes=0)(y_grid)

    nll_grids = jax.vmap(nll_grid_one)(data_reps)  # (N_reps, N_grid1, N_grid2)

    flat = nll_grids.reshape(N_reps, -1)
    idx = jnp.argmin(flat, axis=1)

    i1 = idx // nll_grids.shape[2]
    i2 = idx %  nll_grids.shape[2]

    return key_next, i1, i2


###################################
### Run

for i, MB in enumerate(MBs):
     
    outd = outd_main + f'{MB[0].name}_{MB[1].name}/'
    os.makedirs(outd + 'hats', exist_ok=True)
    
    pars['mat1'] = MB[0]
    pars['mat2'] = MB[1]
    
    print(f'{MB[0].name} / {MB[1].name} basis')

    # ---------------- Build MB-specific (T1, T2) grid ----------------
    N_grid1 = [15, 11][i]
    N_grid2 = [11, 11][i]
    dT1 = DTYPE([1e-6, 4e-6][i])
    dT2 = DTYPE([5e-6, 10e-6][i])

    T1_0 = DTYPE(pars['T1'])
    T2_0 = DTYPE(pars['T2'])

    T1_lim = (T1_0 - dT1 * (N_grid1 / 2), T1_0 + dT1 * (N_grid1 / 2))
    T2_lim = (T2_0 - dT2 * (N_grid2 / 2), T2_0 + dT2 * (N_grid2 / 2))

    T1s = jnp.linspace(T1_lim[0], T1_lim[1], N_grid1, dtype=JTYPE)
    T2s = jnp.linspace(T2_lim[0], T2_lim[1], N_grid2, dtype=JTYPE)
    T1_grid, T2_grid = jnp.meshgrid(T1s, T2s)

    np.array(T1s).astype(DTYPE).tofile(outd+'T1s.bin')
    np.array(T2s).astype(DTYPE).tofile(outd+'T2s.bin')

    # ---------------- Iterate over thresholds ----------------
    mle_vars1, mle_vars2 = np.zeros((2, len(threshs)), dtype=DTYPE)

    t0 = time()
    for ii, Et in enumerate(threshs):

        spec1, spec2 = get_specs_truncated(energies, counts, Et)
        spec1 = jnp.asarray(spec1, dtype=JTYPE)
        spec2 = jnp.asarray(spec2, dtype=JTYPE)

        model = SpectralPBI(spectrum1=spec1, spectrum2=spec2, material_basis=MB)
        vars_ = model.init(key_init, JTYPE(0.0), JTYPE(0.0))

        f = lambda T1, T2: model.apply(vars_, T1, T2)
        forward = jax.jit(jax.vmap(f, in_axes=(0, 0)))

        data_ideal = forward(T1_gt, T2_gt).squeeze().astype(JTYPE)

        y_grid_flat = forward(T1_grid.ravel(), T2_grid.ravel()).astype(JTYPE)
        y_grid = y_grid_flat.reshape(T2s.size, T1s.size, *y_grid_flat.shape[1:])
        y_grid = jnp.swapaxes(y_grid, 0, 1)

        T1_hats_tot = np.zeros(N_batches * N_reps, dtype=DTYPE)
        T2_hats_tot = np.zeros(N_batches * N_reps, dtype=DTYPE)
        
        for kk in range(N_batches):            
            key_noise, i1s, i2s = mle_batch(key_noise, y_grid, data_ideal)
    
            T1_hats = np.array(T1s[i1s], dtype=DTYPE)
            T2_hats = np.array(T2s[i2s], dtype=DTYPE)

            T1_hats_tot[kk*N_reps:(kk+1)*N_reps] = T1_hats
            T2_hats_tot[kk*N_reps:(kk+1)*N_reps] = T2_hats
            
            np.array(T1_hats).astype(DTYPE).tofile(outd+f'hats/T1s_{Et:.1f}_{N_reps}_{kk}.bin')
            np.array(T2_hats).astype(DTYPE).tofile(outd+f'hats/T2s_{Et:.1f}_{N_reps}_{kk}.bin')

        mle_vars1[ii] = np.var(T1_hats_tot).astype(DTYPE)
        mle_vars2[ii] = np.var(T2_hats_tot).astype(DTYPE)

        print(f'{float(Et):.1f} keV --- {time() - t0:.1f} s')

    np.array(mle_vars1).astype(DTYPE).tofile(outd+'mle_vars1.bin')
    np.array(mle_vars2).astype(DTYPE).tofile(outd+'mle_vars2.bin')
    
    # ---------------- Plot ----------------
    fig, ax = plt.subplots(1, 2, figsize=[7, 2.5], layout='constrained', dpi=300)
    fig.suptitle(f'{MB[0].name} / {MB[1].name} basis')

    for j, mle_vars in enumerate([mle_vars1, mle_vars2]):
        x, y = threshs, mle_vars
        w = int(4 / float(threshs[1]-threshs[0])) + 1
        y_ma = np.convolve(np.pad(y, (w//2, w//2), mode='reflect'), np.ones(w)/w, mode='valid')
        ax[j].plot(x[:len(y_ma)], y_ma[:len(x)], 'b-')  # moving avg
        ax[j].plot(threshs, mle_vars, '.', alpha=0.2)   # dots

    plt.show()
    

# 3. Propagation distance/PSF optimization

In [ ]:
%%time

###################################
### Parameters

E1 = 15
E2 = 25
pars['psf'] = 'gaussian'

outd = f'outputs/de_propdists/{pars["psf"]}_{E1:.0f}_{E2:.0f}/'
os.makedirs(outd, exist_ok=True)

# Generate FWHMs based on magnification function
R1 = 50e-3
propdists = 1e-3 * np.arange(0, 101, 1, dtype=DTYPE)
mags = (R1+propdists) / R1
src_fwhms = 3 * (fov/NX) * (mags-1)
tot_fwhms = np.sqrt(src_fwhms**2 + (2*pars['dx'])**2)  # combine source and det PSF
print(f'Src FWHM range: {1e6*src_fwhms.min():.2f} um -- {1e6*src_fwhms.max():.2f} um')

# Save 
propdists.astype(DTYPE).tofile(outd+'propdists.bin')
src_fwhms.astype(DTYPE).tofile(outd+'fwhm_src.bin')
tot_fwhms.astype(DTYPE).tofile(outd+'fwhm_total.bin')


###################################
### Run

fwhm_cases = {
    'fixed': np.full_like(propdists, tot_fwhms[0]),
    'magnified': tot_fwhms,
}

for j, MB in enumerate(MBs):  

    pars['mat1'] = MB[0]
    pars['mat2'] = MB[1]

    ### Init test plot
    fig, ax = plt.subplots(1, 2, figsize=[8,2.5], dpi=300, layout='constrained')
    fig.suptitle(f'{MB[0].name} / {MB[1].name} basis')

    ### Run
    for fwhm_name, fwhm_vals in fwhm_cases.items():

        crlb_Rs = np.zeros((2, propdists.size), dtype=DTYPE)

        for jj, R in enumerate(propdists):
            pars['R'] = R
            pars['fwhm'] = fwhm_vals[jj]
            cvars, csnrs = crlb.compute_crlb_mono(E1=E1, E2=E2, I0_per_m2=I0_per_m2, **pars)
            crlb_Rs[:, jj] = cvars

        crlb_Rs.astype(DTYPE).tofile(
            outd + f'crlb_{MB[0].name}_{MB[1].name}_{fwhm_name}.bin'
        )

        ### Plot
        for i in range(2):
            ax[i].plot(propdists, crlb_Rs[i], label=fwhm_name)
            ax[i].set_yscale('log')
    ax[1].legend()
    plt.show()
    

# 4. Dual-energy CRLB: $N_x \rightarrow 1$ vs. Roessl 

In [ ]:
def get_crlb_roessl(energies, I0, t1, t2, mat1, mat2):
    counts = I0 * np.ones(energies.size)
    _, beta1 = mat1.delta_beta(energies)
    _, beta2 = mat2.delta_beta(energies)
    mu1 = (4 * np.pi * beta1) / get_wavelen(energies)
    mu2 = (4 * np.pi * beta2) / get_wavelen(energies)

    fim = np.zeros([2,2])
    for j in range(len(energies)):
        I = counts[j] * np.exp( -mu1[j]*t1 - mu2[j]*t2 )
        dI_dt1 = -mu1[j]*I
        dI_dt2 = -mu2[j]*I
        fim[0,0] += dI_dt1**2 / I
        fim[1,0] += dI_dt1 * dI_dt2 / I
        fim[1,1] += dI_dt2**2 / I
    fim[0,1] = fim[1,0]  # symmetry

    return np.linalg.inv(fim).diagonal()

In [ ]:
%%time

outd = 'outputs/de_validation/'
os.makedirs(outd, exist_ok=True)


###################################
### Parameters

NX_all = np.arange(1, 6)

pars['R'] = 0
pars['psf'] = None
pars['fwhm'] = None

I0_per_px = I0_per_m2 * pars['dx']**2  

# Energy sweep
E_LOW = 15
E_HIs = np.arange(E_LOW+1, 33, 1)

# Save 
E_HIs.astype(DTYPE).tofile(outd+'energies.bin')
NX_all.astype(DTYPE).tofile(outd+'Nx.bin')


###################################
### Run

for MB in MBs:
    pars['mat1'] = MB[0]
    pars['mat2'] = MB[1]
        
    var_roessl = np.zeros([2, E_HIs.size])
    var_xpci = np.zeros([NX_all.size, 2, E_HIs.size])
    
    for e, E_HI in enumerate(E_HIs):
        
        energies = np.array([E_LOW, E_HI])
        
        var_roessl[:,e] = get_crlb_roessl(
            energies, I0_per_px, pars['T1'], pars['T1'], pars['mat1'], pars['mat2']
        )

        for j, NX in enumerate(NX_all):
     
            pars['Nx'] = NX  # update Nx!
        
            cvars, csnrs = crlb.compute_crlb_mono(E1=E_LOW, E2=E_HI, I0_per_m2=I0_per_m2, **pars)
            var_xpci[j,:,e] = cvars

    var_roessl.astype(DTYPE).tofile(outd+f'crlb_{MB[0].name}_{MB[1].name}_roessl.bin')
    var_xpci.astype(DTYPE).tofile(outd+f'crlb_{MB[0].name}_{MB[1].name}_xpci.bin')

    # ---------------- plot ----------------
    fig, ax = plt.subplots(1, 2, figsize=[7, 2.5], layout='constrained', dpi=300)
    fig.suptitle(f'{MB[0].name} / {MB[1].name} basis')
    for i in range(2):
        ax[i].set_yscale('log')
        ax[i].plot(E_HIs, var_roessl[i], 'k.--', ms=10, lw=2)
        for j, NX in enumerate(NX_all):
            ax[i].plot(E_HIs, var_xpci[j,i], '-', label=NX)
    ax[0].legend(fontsize=8, title=r'$N_x$', loc='upper right')
    plt.show()
    